In [1]:
import pandas as pd
import numpy as np
from transformers import pipeline

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("../Outputs/merged_text_905.csv")
print(f"Loaded: {df.shape[0]} rows")
print(f"Posts with usable text: {df['has_text'].sum()}")

Loaded: 905 rows
Posts with usable text: 797


In [3]:
# Single binary classifier: toxic vs neutral
tox_model = pipeline(
    "text-classification",
    model="s-nlp/roberta_toxicity_classifier",
    top_k=None,
    truncation=True
)

# Quick test
print(tox_model("I'm so excited for the Super Bowl!"))
print(tox_model("I hate you so much, you idiot"))

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 30429.71it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: s-nlp/roberta_toxicity_classifier
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/local/python/3.12.1/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/local/python/3.12.1/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/

[[{'label': 'neutral', 'score': 0.999958872795105}, {'label': 'toxic', 'score': 4.112465103389695e-05}]]
[[{'label': 'toxic', 'score': 0.9996007084846497}, {'label': 'neutral', 'score': 0.00039935664972290397}]]


In [4]:
texts = df.loc[df["has_text"], "text_for_analysis"].tolist()
texts = [t[:2000] for t in texts]

print(f"Running s-nlp toxicity model on {len(texts)} posts...")
results = tox_model(texts, batch_size=16)
print(f"Got {len(results)} results")

Running s-nlp toxicity model on 797 posts...
Got 797 results


In [5]:
# Drop existing columns if they exist (safe re-run)
cols_to_drop = ["snlp_toxic", "snlp_neutral", "snlp_is_toxic", "snlp_label"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# Extract toxic and neutral scores
toxic_scores = []
neutral_scores = []
for res in results:
    score_dict = {item["label"]: item["score"] for item in res}
    toxic_scores.append(score_dict.get("toxic", 0.0))
    neutral_scores.append(score_dict.get("neutral", 0.0))

snlp_df = pd.DataFrame({
    "snlp_toxic": toxic_scores,
    "snlp_neutral": neutral_scores
})
snlp_df.index = df.index[df["has_text"]]
df = df.join(snlp_df)

# Apply 0.5 threshold
THRESHOLD = 0.5
df["snlp_is_toxic"] = None
df["snlp_label"] = None

mask = df["has_text"]
df.loc[mask, "snlp_is_toxic"] = df.loc[mask, "snlp_toxic"] > THRESHOLD
df.loc[mask & (df["snlp_toxic"] > THRESHOLD), "snlp_label"] = "toxic"
df.loc[mask & (df["snlp_toxic"] <= THRESHOLD), "snlp_label"] = "not_toxic"

print(df[["student_id", "text_source", "snlp_label", "snlp_toxic"]].head())

  student_id         text_source snlp_label  snlp_toxic
0        1_A  caption+transcript  not_toxic    0.000043
1        1_A  caption+transcript  not_toxic    0.000100
2        1_A        caption_only  not_toxic    0.000037
3        2_A        caption_only  not_toxic    0.000044
4        2_A        caption_only  not_toxic    0.002357


In [6]:
print("=== s-nlp toxicity label counts ===")
print(df["snlp_label"].value_counts(dropna=False))
print()
print("=== Toxic posts (score > 0.5) ===")
toxic_only = df[df["snlp_is_toxic"] == True]
print(f"Total toxic: {len(toxic_only)} / {df['has_text'].sum()} usable posts")
print()
print("=== Mean toxic score across all usable posts ===")
print(f"  All usable: {df.loc[df['has_text'], 'snlp_toxic'].mean():.4f}")
if len(toxic_only) > 0:
    print(f"  Toxic posts only: {toxic_only['snlp_toxic'].mean():.4f}")

=== s-nlp toxicity label counts ===
snlp_label
not_toxic    781
None         108
toxic         16
Name: count, dtype: int64

=== Toxic posts (score > 0.5) ===
Total toxic: 16 / 797 usable posts

=== Mean toxic score across all usable posts ===
  All usable: 0.0204
  Toxic posts only: 0.9260


In [7]:
print("=== Toxic posts by text source (s-nlp) ===")
print(pd.crosstab(df["text_source"], df["snlp_is_toxic"]))
print()
print("=== Toxic score by text source ===")
print(df.groupby("text_source")["snlp_toxic"].agg(["mean", "median", "max", "count"]))

=== Toxic posts by text source (s-nlp) ===
snlp_is_toxic       False  True 
text_source                     
caption+transcript    129     14
caption_only          650      2
transcript_only         2      0

=== Toxic score by text source ===
                        mean    median       max  count
text_source                                            
caption+transcript  0.096268  0.000143  0.996578    143
caption_only        0.003845  0.000047  0.990886    652
none                     NaN       NaN       NaN      0
transcript_only     0.010377  0.010377  0.019967      2


In [8]:
toxic_only = df[df["snlp_is_toxic"] == True].sort_values("snlp_toxic", ascending=False)
print(f"=== Top {min(15, len(toxic_only))} toxic posts (s-nlp) ===")
for _, r in toxic_only.head(15).iterrows():
    text_preview = r['text_for_analysis'][:200].replace('\n', ' | ')
    print(f"\n  [{r['snlp_toxic']:.2f}] [{r['student_id']}] [{r['text_source']}]")
    print(f"  {text_preview}")

=== Top 15 toxic posts (s-nlp) ===

  [1.00] [6_A] [caption+transcript]
  dirty blonde core #dirtyblonde #blondehair #naturalhair |  | People die for this, people lie for this, people suck and fuck some guy for this, | pay the toll for this, sell their soul for this, play my part

  [1.00] [12_A] [caption+transcript]
  GYMSKIN doubled his AURA after they didnt BURN THE BEAN ???? #gymskin |  | These are... Too groovy. Look at these. They didn't fucking burn the bean. Look at these fucking...

  [0.99] [6_A] [caption+transcript]
  ok but for real can we all agree that abby would be ilya and cathy would be shane? #abbyleemiller #heatedrivalry #edit #dancemomsedits |  | You suck my d***. Oh, that sounded really bratty. We can figure 

  [0.99] [10_A] [caption_only]
  "pulling up to talk shit with my mom and realizing she's not in the mood to take my side" "Pivot turn exit"

  [0.99] [2_A] [caption+transcript]
  Day one back in the suit.#SpiderManBrandNewDay-in theatres 7.31.26 |  | Alrigh

In [9]:
# Load detoxify output we already saved
detox_df = pd.read_csv("../Outputs/toxicity_detoxify_905.csv")
# Get the boolean is_toxic from detoxify
detox_toxic = detox_df["is_toxic"].fillna(False)

# Compare to s-nlp
snlp_toxic = df["snlp_is_toxic"].fillna(False)

# Agreement matrix
print("=== Detoxify vs s-nlp agreement ===")
print("(rows = Detoxify, columns = s-nlp)")
agreement = pd.crosstab(detox_toxic, snlp_toxic, rownames=["Detoxify_toxic"], colnames=["snlp_toxic"])
print(agreement)
print()
both = (detox_toxic & snlp_toxic).sum()
detox_only = (detox_toxic & ~snlp_toxic).sum()
snlp_only = (~detox_toxic & snlp_toxic).sum()
print(f"Both flag as toxic: {both}")
print(f"Detoxify only: {detox_only}")
print(f"s-nlp only: {snlp_only}")

=== Detoxify vs s-nlp agreement ===
(rows = Detoxify, columns = s-nlp)
snlp_toxic      False  True 
Detoxify_toxic              
False             880      4
True                9     12

Both flag as toxic: 12
Detoxify only: 9
s-nlp only: 4


In [10]:
output_path = "../Outputs/toxicity_sNLP_905.csv"
df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: ../Outputs/toxicity_sNLP_905.csv


# Toxicity Model 2: s-nlp RoBERTa — Conclusion

**Model:** `s-nlp/roberta_toxicity_classifier`
**Output labels:** Binary — toxic vs neutral (one probability score each)
**Threshold applied:** 0.5
**Dataset:** 905 posts from 12 students, 797 had usable text

## What we did in plain terms

We ran every post through a second toxicity model. This one is simpler than Detoxify — it just gives one binary score: is this toxic or not? No subtypes like "insult" or "threat."

## What we found

Out of 797 usable posts, 16 (2.0%) crossed the 0.5 threshold and were labeled toxic. Detoxify flagged 21, so the two models are in similar territory but not identical.

The transcript effect is even stronger here than with Detoxify:
- Caption only: 2 toxic out of 652 (0.3%)
- Caption + transcript: 14 toxic out of 143 (9.8%)

Caption-only posts almost never get flagged by this model. It really needs the transcript audio content to detect toxicity.

## Cross-model agreement (the key result)

We compared the s-nlp flags directly with the Detoxify flags from the previous run:

| | s-nlp: not toxic | s-nlp: toxic |
|---|---|---|
| **Detoxify: not toxic** | 880 | 4 |
| **Detoxify: toxic** | 9 | 12 |

- 12 posts where both models agree: toxic
- 9 posts where only Detoxify flags
- 4 posts where only s-nlp flags
- 880 posts where both agree: clean

Out of 21 Detoxify-flagged posts, only 12 (57%) are confirmed by s-nlp. So roughly half of any single model's "toxic" labels are model-specific noise, not reliable signal.

## How this compares to student self-reports (Q8)

Students themselves flagged 10 posts as uncivil. The consensus set of 12 (both models agree) is very close to that number. When models agree, the count converges with human perception.

| Source | Count |
|---|---|
| Q8 student-flagged uncivil | 10 |
| Detoxify ∩ s-nlp consensus | 12 |
| Detoxify alone | 21 |
| s-nlp alone | 16 |

This suggests a useful methodology: treat model agreement as a confidence filter. Don't trust any single model's toxicity label — only trust labels where multiple models converge.

## What's still wrong

Even the 12 consensus posts are mostly entertainment content with casual profanity, not genuinely harmful material. Both models are detecting profanity, not actually distinguishing harm from casual swearing in gym videos, music lyrics, and pet vlogs.

Both models also got tricked by two duplicate-transcript posts in the dataset (the Trump tariffs and "THE DRAMA April 3rd" posts share a wedding dialogue transcript that doesn't belong to either of them).

## Bottom line

s-nlp gives a simpler output than Detoxify but finds roughly the same set of posts. The interesting result is the cross-model agreement: only 12 posts are flagged by both models, which is much closer to the 10 posts students themselves flagged as uncivil.

For a per-post "toxic vs not" label, the most defensible approach is "toxic only if 2+ models agree." This filters out single-model false positives and yields counts that align with human perception.

But this still doesn't solve the underlying problem: both models confuse profanity with toxicity. To distinguish "casual entertainment swearing" from "actual harassment or hate speech," we likely need LLM-based classification.